In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.metrics.pairwise import (
    cosine_similarity,
    euclidean_distances)

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
EMBEDDING_MODEL = "text-embedding-3-small"




In [2]:
documents = [
    {"id": 1,  "text": "Spring Boot simplifies building production-ready Java backend applications.",                                               "topic": "backend",           "level": "beginner"},
    {"id": 2,  "text": "Kubernetes orchestrates containers and provides scheduling, service discovery, and self-healing.",                          "topic": "devops",            "level": "intermediate"},
    {"id": 3,  "text": "Redis is an in-memory data store commonly used for caching, sessions, and fast lookups.",                                  "topic": "databases",         "level": "beginner"},
    {"id": 4,  "text": "Kafka is a distributed event streaming platform used to build high-throughput asynchronous systems.",                      "topic": "distributed-systems","level": "intermediate"},
    {"id": 5,  "text": "PostgreSQL is a relational database with strong SQL support, transactions, indexing, and extensibility.",                  "topic": "databases",         "level": "beginner"},
    {"id": 6,  "text": "Vector databases store embeddings and make similarity search over high-dimensional vectors efficient.",                    "topic": "ai",                "level": "beginner"},
    {"id": 7,  "text": "RAG retrieves relevant external context and gives that context to a language model before generating an answer.",          "topic": "ai",                "level": "beginner"},
    {"id": 8,  "text": "An embedding model converts text into a dense numerical vector that captures semantic relationships.",                    "topic": "ai",                "level": "beginner"},
    {"id": 9,  "text": "Cosine similarity compares the angle between vectors and is commonly used for semantic text retrieval.",                  "topic": "ai",                "level": "intermediate"},
    {"id": 10, "text": "HNSW builds a navigable graph so approximate nearest-neighbor search can avoid scanning every vector.",                   "topic": "vector-search",     "level": "advanced"},
    {"id": 11, "text": "Exact nearest-neighbor search compares the query against every candidate vector to obtain the true top results.",         "topic": "vector-search",     "level": "intermediate"},
    {"id": 12, "text": "Quantization compresses vector representations so a vector database can reduce memory usage and improve search efficiency.","topic": "vector-search",     "level": "advanced"},
    {"id": 13, "text": "Payload indexes allow a vector database to combine semantic similarity with structured filters.",                         "topic": "vector-search",     "level": "advanced"},
    {"id": 14, "text": "Python is widely used for machine learning, data processing, and building AI applications.",                              "topic": "programming",       "level": "beginner"},
    {"id": 15, "text": "Async programming can improve throughput for applications that spend significant time waiting on network I/O.",           "topic": "programming",       "level": "intermediate"},
    {"id": 16, "text": "API gateways can provide routing, authentication, rate limiting, and observability for microservices.",                   "topic": "distributed-systems","level": "intermediate"},
    {"id": 17, "text": "Microservices split an application into independently deployable services that communicate over APIs or events.",         "topic": "distributed-systems","level": "intermediate"},
    {"id": 18, "text": "Observability combines logs, metrics, and traces to understand the behavior of distributed applications.",               "topic": "devops",            "level": "intermediate"},
    {"id": 19, "text": "A cache can reduce database load and latency when frequently requested data can be reused safely.",                       "topic": "databases",         "level": "beginner"},
    {"id": 20, "text": "Reranking can improve retrieval quality by applying a more expensive relevance model to an initial candidate set.",       "topic": "ai",                "level": "advanced"},
]

texts = [d["text"] for d in documents]
print(f"{len(documents)} documents")
print(texts[0])

20 documents
Spring Boot simplifies building production-ready Java backend applications.


In [3]:
def embed_documents(texts: list[str]) -> list[list[float]]:
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [d.embedding for d in resp.data]

doc_embeddings = embed_documents(texts)

print("Number of embeddings:", len(doc_embeddings))
print("Dimensions per vector:", len(doc_embeddings[0]))
print("Values of first vector:", doc_embeddings[0])

Number of embeddings: 20
Dimensions per vector: 1536
Values of first vector: [-0.00774383544921875, 0.01219940185546875, 0.07513427734375, -0.01904296875, 0.044036865234375, -0.037933349609375, 0.017242431640625, 0.040557861328125, 0.0364990234375, -0.045806884765625, 0.0145416259765625, -0.01324462890625, -0.0350341796875, 0.0031490325927734375, -0.0239410400390625, 0.02874755859375, -0.005222320556640625, -0.050537109375, -0.0243072509765625, 0.00933837890625, -0.00717926025390625, -0.020660400390625, 0.03564453125, -0.03759765625, 0.031890869140625, -0.01265716552734375, -0.0282440185546875, -0.02105712890625, -0.03662109375, -0.07440185546875, -0.041961669921875, -0.032745361328125, -0.0006623268127441406, -0.003414154052734375, 0.0445556640625, -0.043914794921875, 0.04779052734375, -0.01349639892578125, 0.0306549072265625, 0.02105712890625, 0.0112457275390625, 0.030731201171875, 0.00591278076171875, -0.026214599609375, 0.0167083740234375, 0.047882080078125, -0.012939453125, 0.0001

In [4]:
query = "Which programming language is commonly used for AI applications?"

query_embedding = embed_documents([query])   # returns a list with one vector

print("Query:", query)
print("Dimensions:", len(query_embedding[0]))
print("Collection must use size =", len(doc_embeddings[0]))

Query: Which programming language is commonly used for AI applications?
Dimensions: 1536
Collection must use size = 1536


In [5]:
cos = cosine_similarity(query_embedding, doc_embeddings)[0]
l2  = euclidean_distances(query_embedding, doc_embeddings)[0]

print(f"{'Cos':>6}  {'L2':>5}  Document")
print("-" * 70)
for doc, c, d in zip(documents, cos, l2):
    print(f"{c:6.3f}  {d:6.3f}  {doc}")

   Cos     L2  Document
----------------------------------------------------------------------
 0.234   1.238  {'id': 1, 'text': 'Spring Boot simplifies building production-ready Java backend applications.', 'topic': 'backend', 'level': 'beginner'}
 0.146   1.307  {'id': 2, 'text': 'Kubernetes orchestrates containers and provides scheduling, service discovery, and self-healing.', 'topic': 'devops', 'level': 'intermediate'}
 0.166   1.291  {'id': 3, 'text': 'Redis is an in-memory data store commonly used for caching, sessions, and fast lookups.', 'topic': 'databases', 'level': 'beginner'}
 0.210   1.258  {'id': 4, 'text': 'Kafka is a distributed event streaming platform used to build high-throughput asynchronous systems.', 'topic': 'distributed-systems', 'level': 'intermediate'}
 0.215   1.253  {'id': 5, 'text': 'PostgreSQL is a relational database with strong SQL support, transactions, indexing, and extensibility.', 'topic': 'databases', 'level': 'beginner'}
 0.227   1.244  {'id': 6, '

In [6]:
def retrieve(query_emb, doc_embs, docs, top_k=2):
    scores = cosine_similarity(query_emb, doc_embs)[0]

    # pair each document with its score, sort highest first, take top_k
    scored = list(zip(docs, scores))
    # print(scored)
    scored.sort(key=lambda x: x[1], reverse=True)

    return [{"document": doc, "score": float(score)} for doc, score in scored[:top_k]]


print("Top-K results:\n")
for hit in retrieve(query_embedding, doc_embeddings, documents, top_k=4):
    print(f"{hit['score']:.3f}  {hit['document']}")

Top-K results:

0.630  {'id': 14, 'text': 'Python is widely used for machine learning, data processing, and building AI applications.', 'topic': 'programming', 'level': 'beginner'}
0.301  {'id': 15, 'text': 'Async programming can improve throughput for applications that spend significant time waiting on network I/O.', 'topic': 'programming', 'level': 'intermediate'}
0.262  {'id': 7, 'text': 'RAG retrieves relevant external context and gives that context to a language model before generating an answer.', 'topic': 'ai', 'level': 'beginner'}
0.234  {'id': 1, 'text': 'Spring Boot simplifies building production-ready Java backend applications.', 'topic': 'backend', 'level': 'beginner'}


In [7]:
from qdrant_client import QdrantClient, models

# qdrant = QdrantClient(":memory:")

qdrant = QdrantClient(url="http://localhost:6333")

In [8]:
COLLECTION = "rag_fundamentalss"

if qdrant.collection_exists(COLLECTION):
    qdrant.delete_collection(COLLECTION)

qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=models.VectorParams(
        size=len(doc_embeddings[0]),   # always read from the model, never hard-code
        distance=models.Distance.COSINE,
    ),
)

print("Created:", COLLECTION, "| size:", len(doc_embeddings[0]))

Created: rag_fundamentalss | size: 1536


In [9]:
points = [
    models.PointStruct(
        id=doc["id"],
        vector=vec,
        payload={"text": doc["text"], "topic": doc["topic"], "level": doc["level"]},
    )
    for doc, vec in zip(documents, doc_embeddings)
]

qdrant.upsert(collection_name=COLLECTION, points=points, wait=True)
print(f"Inserted {len(points)} points.")

Inserted 20 points.


In [10]:
results = qdrant.query_points(
    collection_name=COLLECTION,
    query=query_embedding[0],    # a single vector, not a list
    limit=5,
    with_payload=True,
).points

for p in results:
    print(p)

id=14 version=1 score=0.62990963 payload={'text': 'Python is widely used for machine learning, data processing, and building AI applications.', 'topic': 'programming', 'level': 'beginner'} vector=None shard_key=None order_value=None
id=15 version=1 score=0.3005479 payload={'text': 'Async programming can improve throughput for applications that spend significant time waiting on network I/O.', 'topic': 'programming', 'level': 'intermediate'} vector=None shard_key=None order_value=None
id=7 version=1 score=0.2616669 payload={'text': 'RAG retrieves relevant external context and gives that context to a language model before generating an answer.', 'topic': 'ai', 'level': 'beginner'} vector=None shard_key=None order_value=None
id=1 version=1 score=0.23416296 payload={'text': 'Spring Boot simplifies building production-ready Java backend applications.', 'topic': 'backend', 'level': 'beginner'} vector=None shard_key=None order_value=None
id=17 version=1 score=0.22802225 payload={'text': 'Micro

In [11]:
compressedClient = qdrant.create_collection(
    collection_name="compressed2",
    vectors_config=models.VectorParams(size=len(doc_embeddings[0]), distance=models.Distance.COSINE),
    quantization_config=models.ScalarQuantization(
        scalar=models.ScalarQuantizationConfig(
            type=models.ScalarType.INT8,   # float32 -> 8-bit integers
            always_ram=True,
        )
    ),
)

In [12]:
qdrant.upsert(collection_name="compressed2", points=points, wait=True)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [13]:
results = qdrant.query_points(
    collection_name="compressed2",
    query=query_embedding[0],
    limit=5,
    with_payload=True,
    with_vectors=True,
).points

print("First 5 values of vector:", results[0].vector[:5])

First 5 values of vector: [-0.011105298, -0.014049422, 0.017100329, 0.014598585, 0.04246861]
